# Setup

In [1]:
import base64, os, imageio
from openai import OpenAI
from PIL import Image

In [2]:
%cd /content/drive/MyDrive/GenAI/OpenAI/Image Generation

/content/drive/MyDrive/GenAI/OpenAI/Image Generation


In [3]:
# Set OpenAI API key in the environment
from google.colab import userdata
openai_api = userdata.get('genai_course')
os.environ['OPENAI_API_KEY'] = openai_api
client = OpenAI()

# Generate Image

In [7]:
# Generate an image
img = client.images.generate(
    model = "gpt-image-1",
    prompt = "a beer bottle that is the fruit of an orange tree",
    n = 1,
    size = "1024x1024"
)

In [13]:
# Show the image output
image_bytes = base64.b64decode(img.data[0].b64_json)
with open("image.png", "wb") as f:
    f.write(image_bytes)

In [14]:
img

ImagesResponse(created=1748007318, data=[Image(b64_json='...', revised_prompt=None, url=None)], usage=Usage(input_tokens=17, input_tokens_details=UsageInputTokensDetails(image_tokens=0, text_tokens=17), output_tokens=4160, total_tokens=4177))

# Image Edit

In [16]:
# Setting the configurations
input_photo = "me.JPG"
prompt_text = "Recreate this image exactly as it is. Don't change a thing"
frames_dir = "frames"
gif_name = "me_edited.gif"
num_loops = 5
size = "1024x1024"
quality = "auto"
model = "gpt-image-1"
n_variations = 1

os.makedirs(frames_dir, exist_ok=True)
current_file = input_photo
saved_paths = []

In [18]:
# Edit the image with OpenAI
edit = client.images.edit(
    model = model,
    image = open(current_file, "rb"),
    prompt = prompt_text,
    size = size,
    quality=quality,
    n = n_variations)

In [19]:
# Decode the output that is b64
image_bytes = base64.b64decode(edit.data[0].b64_json)

# Define the path
output_path = os.path.join(frames_dir, f"frame_{1}.png")
with open(output_path, "wb") as f:
    f.write(image_bytes)

In [20]:
# Build a loop to edit the image 100 times over
for i  in range(1, num_loops + 1):
  print(f"Generating frame {i} out of {num_loops}")
  edit = client.images.edit(
    model = model,
    image = open(current_file, "rb"),
    prompt = prompt_text,
    size = size,
    quality=quality,
    n = n_variations)

  # Store the output in the frames dir
  image_bytes = base64.b64decode(edit.data[0].b64_json)
  output_path = os.path.join(frames_dir, f"frame_{i + 1}.png")
  with open(output_path, "wb") as f:
    f.write(image_bytes)

  # Store the path
  saved_paths.append(output_path)
  current_file = output_path

Generating frame 1 out of 5
Generating frame 2 out of 5
Generating frame 3 out of 5
Generating frame 4 out of 5


In [22]:
# Build a GIF
images = [Image.open(p) for p in saved_paths]
images[0].save(
    gif_name,
    save_all=True,
    append_images=images[1:],
    duration=300, # ms per frame
    loop=0,
    optimize = True
)
print(f"GIF saved at {os.getcwd()}/{gif_name}")

GIF saved at /content/drive/MyDrive/GenAI/OpenAI/Image Generation/me_edited.gif
